In [ ]:
from curl_cffi import requests
import pandas as pd
import time
from datetime import datetime

# =========================================================
# CONFIG
# =========================================================
TOLERANCE = 0.05

print("=" * 100)
print("         LIVE NSE BREAKOUT SCANNER (OPEN = LOW / OPEN = HIGH)")
print("=" * 100)

# =========================================================
# MARKET TIME CHECK
# =========================================================
now = datetime.now().time()
market_start = datetime.strptime("09:15", "%H:%M").time()
market_end = datetime.strptime("15:30", "%H:%M").time()

print(f"\nCurrent Time : {now}")
if not (market_start <= now <= market_end):
    print("\n⚠️ Market closed or outside live hours. NSE data may be empty or static.\n")

print("=" * 100)

# =========================================================
# NSE SESSION WITH BROWSER IMPERSONATION
# =========================================================
# Using curl_cffi to bypass Akamai Bot Manager (NSE's anti-bot system)
session = requests.Session(impersonate="chrome120")

headers = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept": "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer": "https://www.nseindia.com/",
    "Origin": "https://www.nseindia.com",
    "Connection": "keep-alive"
}
session.headers.update(headers)

print("\nInitializing NSE Session (Bypassing Anti-Bot protections)...")

# Warmup calls to establish session cookies
try:
    session.get("https://www.nseindia.com", timeout=10)
    time.sleep(1)
    session.get("https://www.nseindia.com/market-data/live-equity-market", timeout=10)
    time.sleep(1)
    print("✅ NSE Session Ready")
except Exception as e:
    print(f"⚠️ Warning during warmup: {e}")

print("=" * 100)

# =========================================================
# KEYWORD → NSE INDEX MAP
# =========================================================
keyword_to_nse = {
    "NIFTY_AUTO": "NIFTY AUTO",
    "NIFTY_CONSR_DURBL": "NIFTY CONSUMER DURABLES",
    "NIFTY_MEDIA": "NIFTY MEDIA",
    "NIFTY_REALTY": "NIFTY REALTY",
    "NIFTY_CONSUMPTION": "NIFTY CONSUMPTION",
    "NIFTY_PVT_BANK": "NIFTY PRIVATE BANK",
    "NIFTY_BANK": "NIFTY BANK",
    "NIFTY_FMCG": "NIFTY FMCG",
    "NIFTY_FIN_SERVICE": "NIFTY FINANCIAL SERVICES 25/50",
    "NIFTY_ENERGY": "NIFTY ENERGY",
    "NIFTY_PSU_BANK": "NIFTY PSU BANK",
    "NIFTY_METAL": "NIFTY METAL",
    "NIFTY_PHARMA": "NIFTY PHARMA",
    "NIFTY_OIL_AND_GAS": "NIFTY OIL & GAS",
    "NIFTY_IT": "NIFTY IT"
}

# =========================================================
# FETCH SECTOR STRENGTH
# =========================================================
print("\n📡 Fetching Sector Strength...\n")

try:
    sector_url = "https://intradayscreener.com/api/indices/sectorData/1"
    sector_headers = {
        "User-Agent": "Mozilla/5.0",
        "Accept": "application/json",
        "Referer": "https://intradayscreener.com/"
    }

    r_sector = requests.get(sector_url, headers=sector_headers, timeout=10)
    data = r_sector.json()

    df_sector = pd.DataFrame({
        "Sector": data.get("labels", []),
        "Keyword": data.get("keywords", []),
        "SectorChange": data.get("datasets", [])
    })

    df_sector["SectorChange"] = pd.to_numeric(
        df_sector["SectorChange"],
        errors="coerce"
    )

    print(
        df_sector
        .sort_values("SectorChange", ascending=False)
        .to_string(index=False)
    )

except Exception as e:
    print("\n❌ Sector Fetch Error :", e)
    exit()

print("\n" + "=" * 100)

# =========================================================
# FILTER SECTORS
# =========================================================
selected = df_sector[
    abs(df_sector["SectorChange"]) >= 0.5
]

print("\n🔥 SELECTED SECTORS\n")
print(selected.to_string(index=False))
print("\n" + "=" * 100)

# =========================================================
# FETCH INDEX STOCKS
# =========================================================
def fetch_index(index_name):
    encoded = requests.utils.quote(index_name)
    url = f"https://www.nseindia.com/api/equity-stock-indices?index={encoded}"

    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            print(f"{index_name} -> HTTP {r.status_code}")
            return pd.DataFrame()

        data = r.json()
        df = pd.DataFrame(data.get("data", []))
        return df

    except Exception as e:
        print(f"{index_name} Error: {e}")
        return pd.DataFrame()

# =========================================================
# MAIN SCANNER
# =========================================================
print("\n🚀 SCANNING LIVE BREAKOUTS...\n")

for _, sector_row in selected.iterrows():
    keyword = sector_row["Keyword"]
    if keyword not in keyword_to_nse:
        continue

    index_name = keyword_to_nse[keyword]
    trend = "Bullish" if sector_row["SectorChange"] > 0 else "Bearish"

    print("\n" + "=" * 100)
    print(f"{index_name} ({trend})")
    print("=" * 100)

    # =====================================================
    # FETCH LIVE STOCKS
    # =====================================================
    df = fetch_index(index_name)

    if df.empty:
        print("No stock data fetched or blocked.")
        continue

    # Remove index row itself
    df = df[df["symbol"] != index_name]

    # =====================================================
    # NUMERIC CONVERSION
    # =====================================================
    numeric_cols = ["open", "dayHigh", "dayLow", "lastPrice", "pChange", "totalTradedVolume"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # =====================================================
    # BREAKOUT LOGIC
    # =====================================================
    bullish = df[abs(df["open"] - df["dayLow"]) <= TOLERANCE].copy()
    bearish = df[abs(df["open"] - df["dayHigh"]) <= TOLERANCE].copy()

    # =====================================================
    # TREND FILTER
    # =====================================================
    if trend == "Bullish":
        breakout_df = bullish
        breakout_df["Signal"] = "BUY"
    else:
        breakout_df = bearish
        breakout_df["Signal"] = "SELL"

    # =====================================================
    # OUTPUT
    # =====================================================
    if breakout_df.empty:
        print("No breakout stocks found")
        continue

    output_cols = ["symbol", "open", "dayHigh", "dayLow", "lastPrice", "pChange", "totalTradedVolume", "Signal"]
    output_cols = [c for c in output_cols if c in breakout_df.columns]
    
    breakout_df = breakout_df[output_cols]
    breakout_df = breakout_df.sort_values("pChange", ascending=False)

    print(breakout_df.to_string(index=False))

print("\n" + "=" * 100)
print("✅ SCAN COMPLETE")
print("=" * 100)

In [ ]:
# from curl_cffi import requests
# import pandas as pd
# import time
# from datetime import datetime

# # =========================================================
# # CONFIG
# # =========================================================
# TOLERANCE = 0.05

# print("=" * 120)
# print("LIVE NSE BREAKOUT SCANNER WITH INDEX MEMBERSHIP")
# print("=" * 120)

# # =========================================================
# # MARKET TIME CHECK
# # =========================================================
# now = datetime.now().time()
# market_start = datetime.strptime("09:15", "%H:%M").time()
# market_end = datetime.strptime("15:30", "%H:%M").time()

# print(f"\nCurrent Time : {now}")

# if not (market_start <= now <= market_end):
#     print("\n⚠️ Market closed or outside live hours. NSE data may be empty or static.\n")

# # =========================================================
# # NSE SESSION
# # =========================================================
# session = requests.Session(impersonate="chrome120")

# headers = {
#     "User-Agent": (
#         "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
#         "AppleWebKit/537.36 (KHTML, like Gecko) "
#         "Chrome/120.0.0.0 Safari/537.36"
#     ),
#     "Accept": "*/*",
#     "Accept-Language": "en-US,en;q=0.9",
#     "Referer": "https://www.nseindia.com/",
#     "Origin": "https://www.nseindia.com",
#     "Connection": "keep-alive"
# }

# session.headers.update(headers)

# try:
#     session.get("https://www.nseindia.com", timeout=10)
#     time.sleep(1)

#     session.get(
#         "https://www.nseindia.com/market-data/live-equity-market",
#         timeout=10
#     )
#     time.sleep(1)

#     print("✅ NSE Session Ready")

# except Exception as e:
#     print(f"⚠️ Warning during warmup: {e}")

# # =========================================================
# # SECTOR INDICES
# # =========================================================
# keyword_to_nse = {
#     "NIFTY_AUTO": "NIFTY AUTO",
#     "NIFTY_CONSR_DURBL": "NIFTY CONSUMER DURABLES",
#     "NIFTY_MEDIA": "NIFTY MEDIA",
#     "NIFTY_REALTY": "NIFTY REALTY",
#     "NIFTY_CONSUMPTION": "NIFTY CONSUMPTION",
#     "NIFTY_PVT_BANK": "NIFTY PRIVATE BANK",
#     "NIFTY_BANK": "NIFTY BANK",
#     "NIFTY_FMCG": "NIFTY FMCG",
#     "NIFTY_FIN_SERVICE": "NIFTY FINANCIAL SERVICES 25/50",
#     "NIFTY_ENERGY": "NIFTY ENERGY",
#     "NIFTY_PSU_BANK": "NIFTY PSU BANK",
#     "NIFTY_METAL": "NIFTY METAL",
#     "NIFTY_PHARMA": "NIFTY PHARMA",
#     "NIFTY_OIL_AND_GAS": "NIFTY OIL & GAS",
#     "NIFTY_IT": "NIFTY IT"
# }

# # =========================================================
# # FETCH INDEX DATA
# # =========================================================
# def fetch_index(index_name):

#     encoded = requests.utils.quote(index_name)

#     url = (
#         f"https://www.nseindia.com/api/"
#         f"equity-stock-indices?index={encoded}"
#     )

#     try:

#         r = session.get(url, timeout=15)

#         if r.status_code != 200:
#             print(f"{index_name} -> HTTP {r.status_code}")
#             return pd.DataFrame()

#         data = r.json()

#         return pd.DataFrame(data.get("data", []))

#     except Exception as e:
#         print(f"{index_name} Error : {e}")
#         return pd.DataFrame()

# # =========================================================
# # NIFTY MEMBERSHIP MAP
# # =========================================================
# # NSE_MEMBERSHIP_INDICES = {
# #     "NIFTY50": "NIFTY 50",
# #     "NIFTY100": "NIFTY 100",
# #     "NIFTY150": "NIFTY MIDCAP 150",
# #     "NIFTY200": "NIFTY 200",
# #     "NIFTY500": "NIFTY 500"
# # }
# NSE_MEMBERSHIP_INDICES = {
#     "NIFTY50": "NIFTY 50",
#     "NIFTYNEXT50": "NIFTY NEXT 50",
#     "NIFTY100": "NIFTY 100",
#     "NIFTY200": "NIFTY 200",
#     "NIFTY500": "NIFTY 500",
#     "MIDCAP100": "NIFTY MIDCAP 100",
#     "SMALLCAP100": "NIFTY SMALLCAP 100",
#     "TOTALMARKET": "NIFTY TOTAL MARKET"
# }

# print("\n📌 Building Membership Maps...\n")

# membership_map = {}

# for short_name, index_name in NSE_MEMBERSHIP_INDICES.items():

#     print(f"Fetching {index_name}")

#     idx_df = fetch_index(index_name)

#     if idx_df.empty:
#         continue

#     idx_df = idx_df[idx_df["symbol"] != index_name]

#     for stock in idx_df["symbol"]:

#         membership_map.setdefault(stock, {})
#         membership_map[stock][short_name] = "Yes"

# print("\n✅ Membership Maps Ready")

# # =========================================================
# # MAIN SCAN
# # =========================================================
# all_results = []

# print("\n🚀 SCANNING ALL SECTOR STOCKS...\n")

# for keyword, index_name in keyword_to_nse.items():

#     print(f"\nScanning : {index_name}")

#     df = fetch_index(index_name)

#     if df.empty:
#         continue

#     df = df[df["symbol"] != index_name]

#     numeric_cols = [
#         "open",
#         "dayHigh",
#         "dayLow",
#         "lastPrice",
#         "pChange",
#         "totalTradedVolume"
#     ]

#     for col in numeric_cols:

#         if col in df.columns:
#             df[col] = pd.to_numeric(
#                 df[col],
#                 errors="coerce"
#             )

#     # =====================================================
#     # OPEN = LOW
#     # =====================================================
#     bullish = df[
#         abs(df["open"] - df["dayLow"]) <= TOLERANCE
#     ].copy()

#     bullish["Signal"] = "BUY"

#     # =====================================================
#     # OPEN = HIGH
#     # =====================================================
#     bearish = df[
#         abs(df["open"] - df["dayHigh"]) <= TOLERANCE
#     ].copy()

#     bearish["Signal"] = "SELL"

#     breakout_df = pd.concat(
#         [bullish, bearish],
#         ignore_index=True
#     )

#     if breakout_df.empty:
#         continue

#     breakout_df["Sector"] = index_name

#     # =====================================================
#     # NIFTY MEMBERSHIP
#     # =====================================================
#     for idx_name in NSE_MEMBERSHIP_INDICES.keys():

#         breakout_df[idx_name] = breakout_df["symbol"].apply(
#             lambda x: membership_map.get(
#                 x,
#                 {}
#             ).get(
#                 idx_name,
#                 "No"
#             )
#         )

#     all_results.append(breakout_df)

# # =========================================================
# # FINAL OUTPUT
# # =========================================================
# if all_results:

#     final_df = pd.concat(
#         all_results,
#         ignore_index=True
#     )

#     final_df = final_df.drop_duplicates(
#         subset=["symbol"],
#         keep="first"
#     )

#     final_df = final_df.sort_values(
#         "pChange",
#         ascending=False
#     )

#     output_cols = [
#         "symbol",
#         "Sector",
#         "Signal",
#         "NIFTY50",
#         "NIFTY100",
#         "NIFTY150",
#         "NIFTY200",
#         "NIFTY500",
#         "open",
#         "dayHigh",
#         "dayLow",
#         "lastPrice",
#         "pChange",
#         "totalTradedVolume"
#     ]

#     output_cols = [
#         c for c in output_cols
#         if c in final_df.columns
#     ]

#     print("\n")
#     print("=" * 150)
#     print("BREAKOUT RESULTS")
#     print("=" * 150)

#     print(
#         final_df[output_cols]
#         .to_string(index=False)
#     )

#     final_df[output_cols].to_excel(
#         "C:\\Users\\ravin\\Desktop\\Trading setup\\stock-analyzer-tool\\stock\\intraday_report\\nse_breakout_results.xlsx",
#         index=False
#     )

#     print("\n✅ Excel Saved : nse_breakout_results.xlsx")

# else:

#     print("\n❌ No breakout stocks found")

# print("\n" + "=" * 150)
# print("✅ SCAN COMPLETE")
# print("=" * 150)

LIVE NSE BREAKOUT SCANNER WITH INDEX MEMBERSHIP

Current Time : 09:19:58.497510
✅ NSE Session Ready

📌 Building Membership Maps...

Fetching NIFTY 50
NIFTY 50 Error : Failed to perform, curl: (28) Operation timed out after 15015 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.
Fetching NIFTY NEXT 50
Fetching NIFTY 100
Fetching NIFTY 200
Fetching NIFTY 500
Fetching NIFTY MIDCAP 100
Fetching NIFTY SMALLCAP 100
Fetching NIFTY TOTAL MARKET

✅ Membership Maps Ready

🚀 SCANNING ALL SECTOR STOCKS...


Scanning : NIFTY AUTO

Scanning : NIFTY CONSUMER DURABLES

Scanning : NIFTY MEDIA

Scanning : NIFTY REALTY
NIFTY REALTY Error : Failed to perform, curl: (28) Operation timed out after 15003 milliseconds with 0 bytes received. See https://curl.se/libcurl/c/libcurl-errors.html first for more details.

Scanning : NIFTY CONSUMPTION

Scanning : NIFTY PRIVATE BANK
NIFTY PRIVATE BANK Error : Failed to perform, curl: (28) Operation timed out a

In [4]:
from curl_cffi import requests
import pandas as pd
import time
from datetime import datetime

# =========================================================
# CONFIG
# =========================================================
TOLERANCE       = 0.05
MIN_VOLUME      = 50000
MIN_PRICE       = 50
BUY_MIN_PCHANGE = 0.3    # BUY only if stock already up 0.3%+
SELL_MAX_PCHANGE = -0.1  # SELL only if stock already down 0.1%+
TOP_N_BULLISH   = 5      # Top N bullish sectors to scan for BUY
TOP_N_BEARISH   = 2      # Top N bearish sectors to scan for SELL

OUTPUT_PATH = r"C:\Users\ravin\Desktop\Trading setup\stock-analyzer-tool\stock\intraday_report\nse_breakout_results.xlsx"

print("=" * 120)
print("         LIVE NSE BREAKOUT SCANNER (OPEN = LOW / OPEN = HIGH)")
print("=" * 120)

# =========================================================
# MARKET TIME CHECK
# =========================================================
now = datetime.now().time()
market_start = datetime.strptime("09:15", "%H:%M").time()
market_end   = datetime.strptime("15:30", "%H:%M").time()

print(f"\nCurrent Time : {now}")
print("=" * 120)

if not (market_start <= now <= market_end):
    print("\n⚠️  Market closed or outside live hours. NSE data may be empty or static.\n")

# =========================================================
# NSE SESSION
# =========================================================
print("\nInitializing NSE Session (Bypassing Anti-Bot protections)...")

session = requests.Session(impersonate="chrome120")

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    ),
    "Accept":          "*/*",
    "Accept-Language": "en-US,en;q=0.9",
    "Referer":         "https://www.nseindia.com/",
    "Origin":          "https://www.nseindia.com",
    "Connection":      "keep-alive"
}

session.headers.update(headers)

try:
    session.get("https://www.nseindia.com", timeout=10)
    time.sleep(1)
    session.get("https://www.nseindia.com/market-data/live-equity-market", timeout=10)
    time.sleep(1)
    print("✅ NSE Session Ready")
except Exception as e:
    print(f"⚠️  Warning during warmup: {e}")

print("=" * 120)

# =========================================================
# FETCH INDEX DATA
# =========================================================
def fetch_index(index_name):
    encoded = requests.utils.quote(index_name)
    url = f"https://www.nseindia.com/api/equity-stock-indices?index={encoded}"
    try:
        r = session.get(url, timeout=15)
        if r.status_code != 200:
            print(f"  ⚠️  {index_name} -> HTTP {r.status_code}")
            return pd.DataFrame()
        data = r.json()
        return pd.DataFrame(data.get("data", []))
    except Exception as e:
        print(f"  ⚠️  {index_name} Error : {e}")
        return pd.DataFrame()

# =========================================================
# SECTOR STRENGTH
# =========================================================
SECTOR_INDICES = {
    "IT":                          "NIFTY IT",
    "CONSUMER GOODS":              "NIFTY FMCG",
    "CONSTRUCTION":                "NIFTY REALTY",
    "MEDIA & ENTERTAINMENT":       "NIFTY MEDIA",
    "Nifty Consumer Durables":     "NIFTY CONSUMER DURABLES",
    "Consumption":                 "NIFTY CONSUMPTION",
    "Nifty Financial Services 25/50": "NIFTY FINANCIAL SERVICES 25/50",
    "Nifty Financial Services":    "NIFTY FINANCIAL SERVICES",
    "Energy":                      "NIFTY ENERGY",
    "PHARMA":                      "NIFTY PHARMA",
    "AUTOMOBILE":                  "NIFTY AUTO",
    "PSU Bank":                    "NIFTY PSU BANK",
    "Nifty Healthcare Index":      "NIFTY HEALTHCARE",
    "Bank Nifty":                  "NIFTY BANK",
    "Nifty Oil & Gas":             "NIFTY OIL & GAS",
    "PVT Bank":                    "NIFTY PRIVATE BANK",
    "METALS":                      "NIFTY METAL",
}

print("\n📡 Fetching Sector Strength...\n")

sector_strength = []

for sector_label, index_name in SECTOR_INDICES.items():
    df_idx = fetch_index(index_name)
    if df_idx.empty:
        continue
    # First row is the index itself
    index_row = df_idx[df_idx["symbol"] == index_name]
    if index_row.empty:
        index_row = df_idx.iloc[[0]]
    pchange = pd.to_numeric(index_row["pChange"].values[0], errors="coerce")
    sector_strength.append({
        "Sector":        sector_label,
        "IndexName":     index_name,
        "SectorChange":  round(pchange, 2)
    })
    time.sleep(0.3)

sector_df = pd.DataFrame(sector_strength).sort_values("SectorChange", ascending=False)
print(sector_df[["Sector", "IndexName", "SectorChange"]].to_string(index=False))

# =========================================================
# SELECT TOP BULLISH & BEARISH SECTORS
# =========================================================
bullish_sectors = sector_df.head(TOP_N_BULLISH)
bearish_sectors = sector_df[sector_df["SectorChange"] < 0].tail(TOP_N_BEARISH)

selected_sectors = pd.concat([bullish_sectors, bearish_sectors]).drop_duplicates()

print("\n" + "=" * 120)
print("\n🔥 SELECTED SECTORS\n")
print(selected_sectors[["Sector", "IndexName", "SectorChange"]].to_string(index=False))
print("\n" + "=" * 120)

# =========================================================
# NIFTY MEMBERSHIP MAP
# =========================================================
NSE_MEMBERSHIP_INDICES = {
    "NIFTY50":      "NIFTY 50",
    "NIFTYNEXT50":  "NIFTY NEXT 50",
    "NIFTY100":     "NIFTY 100",
    "NIFTY200":     "NIFTY 200",
    "NIFTY500":     "NIFTY 500",
    "MIDCAP100":    "NIFTY MIDCAP 100",
    "SMALLCAP100":  "NIFTY SMALLCAP 100",
    "TOTALMARKET":  "NIFTY TOTAL MARKET"
}

print("\n📌 Building Membership Maps...\n")

membership_map = {}

for short_name, index_name in NSE_MEMBERSHIP_INDICES.items():
    print(f"  Fetching {index_name}")
    idx_df = fetch_index(index_name)
    if idx_df.empty:
        continue
    idx_df = idx_df[idx_df["symbol"] != index_name]
    for stock in idx_df["symbol"]:
        membership_map.setdefault(stock, {})
        membership_map[stock][short_name] = "Yes"
    time.sleep(0.3)

print("\n✅ Membership Maps Ready")
print("=" * 120)

# =========================================================
# MAIN SCAN
# =========================================================
all_results = []

print("\n🚀 SCANNING LIVE BREAKOUTS...\n")

for _, sector_row in selected_sectors.iterrows():

    index_name    = sector_row["IndexName"]
    sector_change = sector_row["SectorChange"]
    is_bullish    = sector_change >= 0

    print("\n" + "=" * 120)
    direction = "Bullish" if is_bullish else "Bearish"
    print(f"{index_name} ({direction})")
    print("=" * 120)

    df = fetch_index(index_name)
    time.sleep(0.5)

    if df.empty:
        print("No stock data fetched or blocked.")
        continue

    df = df[df["symbol"] != index_name]

    numeric_cols = ["open", "dayHigh", "dayLow", "lastPrice", "pChange", "totalTradedVolume"]
    for col in numeric_cols:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ----------------------------------------------------------
    # FILTER 1: Remove circuit-locked stocks (dayHigh == dayLow)
    # ----------------------------------------------------------
    df = df[df["dayHigh"] != df["dayLow"]]

    # ----------------------------------------------------------
    # FILTER 2: Minimum price (no penny stocks)
    # ----------------------------------------------------------
    df = df[df["lastPrice"] >= MIN_PRICE]

    # ----------------------------------------------------------
    # FILTER 3: Minimum volume (liquidity check)
    # ----------------------------------------------------------
    df = df[df["totalTradedVolume"] >= MIN_VOLUME]

    breakout_df = pd.DataFrame()

    if is_bullish:
        # ======================================================
        # OPEN = LOW  →  BUY signal (only in bullish sectors)
        # ======================================================
        bullish = df[
            (abs(df["open"] - df["dayLow"]) <= TOLERANCE) &
            (df["pChange"] >= BUY_MIN_PCHANGE)   # Must have real momentum
        ].copy()
        bullish["Signal"] = "BUY"
        breakout_df = bullish

    else:
        # ======================================================
        # OPEN = HIGH  →  SELL signal (only in bearish sectors)
        # ======================================================
        bearish = df[
            (abs(df["open"] - df["dayHigh"]) <= TOLERANCE) &
            (df["pChange"] <= SELL_MAX_PCHANGE)  # Must already be falling
        ].copy()
        bearish["Signal"] = "SELL"
        breakout_df = bearish

    if breakout_df.empty:
        print("No breakout stocks found in this sector.")
        continue

    breakout_df["Sector"] = index_name

    # ----------------------------------------------------------
    # NIFTY MEMBERSHIP
    # ----------------------------------------------------------
    for idx_name in NSE_MEMBERSHIP_INDICES.keys():
        breakout_df[idx_name] = breakout_df["symbol"].apply(
            lambda x: membership_map.get(x, {}).get(idx_name, "No")
        )

    all_results.append(breakout_df)

    display_cols = ["symbol", "open", "dayHigh", "dayLow", "lastPrice", "pChange", "totalTradedVolume", "Signal"]
    display_cols = [c for c in display_cols if c in breakout_df.columns]
    print(breakout_df[display_cols].to_string(index=False))

# =========================================================
# FINAL OUTPUT
# =========================================================
print("\n")
print("=" * 150)
print("BREAKOUT RESULTS")
print("=" * 150)

if all_results:

    final_df = pd.concat(all_results, ignore_index=True)
    final_df = final_df.drop_duplicates(subset=["symbol"], keep="first")

    # ----------------------------------------------------------
    # PRIORITY SCORE: Float large-cap quality stocks to top
    # ----------------------------------------------------------
    final_df["Priority"] = (
        (final_df["NIFTY50"]   == "Yes").astype(int) * 4 +
        (final_df["NIFTY100"]  == "Yes").astype(int) * 3 +
        (final_df["NIFTY200"]  == "Yes").astype(int) * 2 +
        (final_df["NIFTY500"]  == "Yes").astype(int) * 1
    )

    # Sort BUY by Priority + pChange desc, SELL by Priority + pChange asc
    buy_df  = final_df[final_df["Signal"] == "BUY"].sort_values(
        ["Priority", "pChange"], ascending=[False, False]
    )
    sell_df = final_df[final_df["Signal"] == "SELL"].sort_values(
        ["Priority", "pChange"], ascending=[False, True]
    )

    final_df = pd.concat([buy_df, sell_df], ignore_index=True)

    output_cols = [
        "symbol", "Sector", "Signal",
        "NIFTY50", "NIFTY100", "NIFTY200", "NIFTY500",
        "open", "dayHigh", "dayLow", "lastPrice",
        "pChange", "totalTradedVolume", "Priority"
    ]
    output_cols = [c for c in output_cols if c in final_df.columns]

    print(final_df[output_cols].to_string(index=False))

    final_df[output_cols].to_excel(OUTPUT_PATH, index=False)
    print(f"\n✅ Excel Saved : {OUTPUT_PATH}")

else:
    print("\n❌ No breakout stocks found.")

print("\n" + "=" * 150)
print("✅ SCAN COMPLETE")
print("=" * 150)

         LIVE NSE BREAKOUT SCANNER (OPEN = LOW / OPEN = HIGH)

Current Time : 09:56:05.494034

Initializing NSE Session (Bypassing Anti-Bot protections)...
✅ NSE Session Ready

📡 Fetching Sector Strength...

                Sector         IndexName  SectorChange
 MEDIA & ENTERTAINMENT       NIFTY MEDIA          1.66
                    IT          NIFTY IT          1.26
                Energy      NIFTY ENERGY          0.74
        CONSUMER GOODS        NIFTY FMCG          0.65
          CONSTRUCTION      NIFTY REALTY          0.36
           Consumption NIFTY CONSUMPTION          0.25
                PHARMA      NIFTY PHARMA          0.18
            Bank Nifty        NIFTY BANK          0.06
Nifty Healthcare Index  NIFTY HEALTHCARE          0.05
              PSU Bank    NIFTY PSU BANK          0.05
            AUTOMOBILE        NIFTY AUTO         -0.07
                METALS       NIFTY METAL         -1.12


🔥 SELECTED SECTORS

               Sector    IndexName  SectorChange
MEDIA 